In [1]:
import os
import random
import torch
from ray.data.preprocessor import Preprocessor
import ray
import numpy as np
import torch.nn.functional as F

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
BERT_MODEL = os.environ["BERT_MODEL"]
BERT_TOKENIZER = os.environ["BERT_TOKENIZER"]

In [4]:
if ray.is_initialized():
    ray.shutdown()
context=ray.init(num_cpus=2)
print(context.dashboard_url)

2026-03-28 15:09:32,160	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


127.0.0.1:8265


c:\Users\admin\Documents\Projects\MadeWithML\.venv\Lib\site-packages\ray\_private\worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [5]:
def set_seed(seed=1980):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    os.environ["PYTHONHASHSEED"] = str(seed)

In [6]:
def load_data(num_samples=None):
    ds=ray.data.read_parquet('local_data/train_v1')
    if num_samples:
        ds=ds.limit(num_samples)
    ds=ds.random_shuffle(seed=1980)
    return ds


In [7]:
import torch.nn as nn
from transformers import BertModel, BertTokenizer

In [8]:
llm=BertModel.from_pretrained(BERT_MODEL,return_dict=False)
embedding_dim=llm.config.hidden_size

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
tokenizer = BertTokenizer.from_pretrained(BERT_TOKENIZER, return_dict=False)
text="Who the hell are you"
batch=tokenizer([text,"Say hello to my friend!"],return_tensors='np',padding='longest')
batch={k:torch.tensor(v) for k,v in batch.items()}
seq, pool = llm(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
np.shape(seq),np.shape(pool)

(torch.Size([2, 9, 768]), torch.Size([2, 768]))

In [10]:
seq,pool

(tensor([[[-1.0321,  0.4903,  0.5299,  ..., -0.0704, -0.2695, -1.3656],
          [-0.8629, -0.6905, -0.2585,  ..., -0.5688, -0.1277, -1.1368],
          [-0.9505,  0.8434,  0.9905,  ..., -0.0283, -1.0176, -1.1304],
          ...,
          [-0.6602,  0.4632,  0.5043,  ..., -0.2438, -0.5053, -1.0504],
          [-1.1808,  0.7772, -0.0784,  ..., -0.0845, -0.5098, -1.1882],
          [-1.2484,  0.7366, -0.1473,  ..., -0.1786, -0.3819, -1.2054]],
 
         [[-1.4089,  0.5280,  1.2531,  ..., -0.0277, -0.1427, -0.7830],
          [-0.6959,  0.5572,  0.8155,  ..., -0.0897, -0.1176, -0.7108],
          [-2.3211,  0.6735,  0.7357,  ...,  0.1255, -0.5972, -0.7836],
          ...,
          [-0.5946,  0.5421,  0.5934,  ...,  0.1477, -0.0682, -1.4599],
          [-0.9842,  0.6600,  1.2748,  ..., -0.0335, -0.2505, -0.6862],
          [-1.1393,  0.6572,  1.2320,  ..., -0.1366, -0.5389, -0.7346]]],
        grad_fn=<NativeLayerNormBackward0>),
 tensor([[ 0.3265,  0.2602,  0.3069,  ..., -0.1927,  0.8

In [11]:
class FinetunedLLM(nn.Module):
    def __init__(self, llm, dropout_p, embedding_dim, num_classes):
        super().__init__()
        self.llm=llm
        self.dropout=nn.Dropout(dropout_p)
        self.fc1=nn.Linear(embedding_dim,num_classes)
    def forward(self,batch):
        ids, masks=batch["ids"],batch["masks"]
        seq, pool =  self.llm(input_ids=ids, attention_mask=masks)
        z=self.dropout(pool)
        z=self.fc1(z)
        return z
    # inference_mode: not tracking gradients
    # self.eval(): disable dropout
    @torch.inference_mode()
    def predict(self,batch):
        self.eval()
        z = self(batch)
        y_pred=torch.argmax(z,dim=1).cpu().numpy()
        return y_pred
    @torch.inference_mode()
    def predict_proba(self,batch):
        self.eval()
        z=self(batch)
        y_probs=F.softmax(z,dim=1).cpu().numpy()
        return y_probs

In [12]:
train_ds=load_data()

# Group by 'targets' and count the number of groups
num_classes = train_ds.groupby("targets").count().count()

Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-03-28 15:10:10,631	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 0.977.
2026-03-28 15:10:10,634	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 214813 rows
2026-03-28 15:10:12,895	INFO logging.py:392 -- Registered dataset logger for dataset dataset_3_0
2026-03-28 15:10:12,943	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=4, aggregators=2, dataset (estimate)=0.0GiB): shuffle=0.2MiB, output=0.1MiB, total=0.3MiB, 
2026-03-28 15:10:12,952	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_3_0. Full logs are in C:\Users\admin\AppData\Local\Temp\ray\session_2026-03-28_15-09-22_830872_9044\logs\ray-data
2026-03-28 15:10:12,953	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> AllToAllOperator[ReadParquet->RandomShuffle] -> HashAggregateOperator[HashAggregate(key_columns=('targets',), num_partitions=4)] -> TaskPoolMapO

In [13]:
model=FinetunedLLM(llm=llm,
                   dropout_p=0.5,
                   embedding_dim=embedding_dim,
                   num_classes=num_classes)
model.parameters()

<generator object Module.parameters at 0x000001F9DE5DAB20>

In [14]:
from ray.train.torch import get_device
def pad_array(arr, dtype=np.int32):
    max_len=max(len(row) for row in arr)
    padded_arr=np.zeros((arr.shape[0],max_len),dtype=dtype)
    for i, row in enumerate(arr):
        padded_arr[i,:len(row)]=row
    return padded_arr

In [15]:

def collate_fn(batch):
    batch["ids"] = pad_array(batch["ids"])
    batch["masks"] = pad_array(batch["masks"])
    dtypes={"ids":torch.int32, 
            "masks":torch.int32,
            "targets":torch.int64}
    try:
        device=get_device()
    except:
        device=None    
    tensor_batch={key:torch.tensor(array,dtype=dtypes[key], device=None) for key, array in batch.items()}
    return tensor_batch

In [16]:
sample_batch=train_ds.take_batch(batch_size=128)
collate_fn(batch=sample_batch)

2026-03-28 15:10:18,049	INFO logging.py:392 -- Registered dataset logger for dataset dataset_4_0
2026-03-28 15:10:18,063	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_4_0. Full logs are in C:\Users\admin\AppData\Local\Temp\ray\session_2026-03-28_15-09-22_830872_9044\logs\ray-data
2026-03-28 15:10:18,065	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> AllToAllOperator[ReadParquet->RandomShuffle] -> LimitOperator[limit=128]
2026-03-28 15:10:18,122	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-03-28 15:10:18,127	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-03-28 15:10:18,130	INFO logging_progress.py:227 -- Active & requested resources: 0/2 CPU, 0.0B/113.4MiB object store
2026-03-28 15:10:18,133	INFO logging_progress.py:181 -- 
2026-03-28 15:10:18,134	INFO logging_progress.py:231 -- ReadParquet->RandomShuffle: 0/1
2026-03-28 15:10:18,135	INFO logging_progress.py:233 

{'ids': tensor([[  102, 19294,  3577,  ...,     0,     0,     0],
         [  102, 27458,   106,  ...,     0,     0,     0],
         [  102,  6116,   140,  ...,     0,     0,     0],
         ...,
         [  102,  2570,  7885,  ...,     0,     0,     0],
         [  102,  2934, 21190,  ...,     0,     0,     0],
         [  102,   466, 15282,  ...,     0,     0,     0]], dtype=torch.int32),
 'masks': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32),
 'targets': tensor([2, 3, 1, 3, 3, 3, 0, 2, 2, 3, 0, 3, 3, 3, 0, 3, 2, 3, 3, 1, 3, 0, 0, 3,
         2, 3, 2, 2, 2, 0, 0, 2, 3, 3, 2, 3, 3, 2, 3, 2, 3, 2, 3, 3, 2, 3, 3, 3,
         2, 2, 1, 2, 3, 3, 2, 3, 0, 3, 3, 2, 3, 0, 2, 2, 0, 1, 1, 2, 2, 3, 2, 3,
         3, 3, 1, 3, 3, 2, 1, 2, 3, 3, 3, 3, 0, 2, 0, 3, 2, 2, 2, 2, 2, 0, 2, 2,
         2, 3, 

In [17]:
from ray.train import CheckpointConfig, RunConfig, ScalingConfig, session, DataConfig, Checkpoint

import ray.train as train
from ray.train.torch import TorchCheckpoint, TorchTrainer

In [18]:
def train_step(ds: ray.data.Dataset, batch_size, model, num_classes, loss_fn, optimizer):
    model.train()
    loss = 0
    ds_generator = ds.iter_torch_batches(batch_size=batch_size,collate_fn=collate_fn)
    for i, batch in enumerate(ds_generator):
        optimizer.zero_grad()
        preds=model(batch)
        J=loss_fn(preds, batch["targets"])
        J.backward()
        optimizer.step()
        loss+=(J.item()-loss)/(i+1) #cumulative loss
    return loss

In [19]:
@torch.inference_mode()
def eval_step(ds: ray.data.Dataset, batch_size, model, num_classes, loss_fn):
    model.eval()
    loss = 0
    y_trues=[]
    y_preds=[]
    ds_generator = ds.iter_torch_batches(batch_size=batch_size,collate_fn=collate_fn)
    for i, batch in enumerate(ds_generator):
        preds=model(batch)
        J=loss_fn(preds, batch["targets"])
        loss+=(J.item()-loss)/(i+1) #cumulative loss
        y_trues.extend(batch["targets"].cpu().numpy())
        y_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
    return loss, np.vstack(y_trues), np.vstack(y_preds)


In [20]:
def train_loop_per_worker(config):
    # FIX 8: all imports done locally inside the worker so Ray doesn't try to
    # serialize top-level torch/cudnn module state, which is not picklable.
    import os
    import random
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from transformers import BertModel
    import ray.train as train
    from ray.train import session
    from ray.train.torch import TorchCheckpoint, get_device

    # ── local helpers ────────────────────────────────────────────────────────

    def set_seed(seed=1980):
        np.random.seed(seed)
        random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ["PYTHONHASHSEED"] = str(seed)

    def pad_array(arr, dtype=np.int32):
        max_len = max(len(row) for row in arr)
        padded_arr = np.zeros((arr.shape[0], max_len), dtype=dtype)
        for i, row in enumerate(arr):
            padded_arr[i, :len(row)] = row
        return padded_arr

    def collate_fn(batch):
        batch["ids"] = pad_array(batch["ids"])
        batch["masks"] = pad_array(batch["masks"])
        dtypes = {"ids": torch.int32, "masks": torch.int32, "targets": torch.int64}
        try:
            device = get_device()
        except Exception:
            device = None
        return {k: torch.tensor(v, dtype=dtypes[k], device=device) for k, v in batch.items()}

    class FinetunedLLM(nn.Module):
        def __init__(self, llm, dropout_p, embedding_dim, num_classes):
            super().__init__()
            self.llm = llm
            self.dropout = nn.Dropout(dropout_p)
            self.fc1 = nn.Linear(embedding_dim, num_classes)

        def forward(self, batch):
            ids, masks = batch["ids"], batch["masks"]
            seq, pool = self.llm(input_ids=ids, attention_mask=masks)
            z = self.dropout(pool)
            return self.fc1(z)

        @torch.inference_mode()
        def predict(self, batch):
            self.eval()
            return torch.argmax(self(batch), dim=1).cpu().numpy()

        @torch.inference_mode()
        def predict_proba(self, batch):
            self.eval()
            return F.softmax(self(batch), dim=1).cpu().numpy()

    def train_step(ds, batch_size, model, num_classes, loss_fn, optimizer):
        model.train()
        loss = 0
        for i, batch in enumerate(ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)):
            optimizer.zero_grad()
            J = loss_fn(model(batch), batch["targets"])
            J.backward()
            optimizer.step()
            loss += (J.item() - loss) / (i + 1)
        return loss

    @torch.inference_mode()
    def eval_step(ds, batch_size, model, num_classes, loss_fn):
        model.eval()
        loss, y_trues, y_preds = 0, [], []
        for i, batch in enumerate(ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)):
            preds = model(batch)
            J = loss_fn(preds, batch["targets"])
            loss += (J.item() - loss) / (i + 1)
            y_trues.extend(batch["targets"].cpu().numpy())
            y_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
        return loss, np.vstack(y_trues), np.vstack(y_preds)

    # ── training setup ───────────────────────────────────────────────────────

    dropout_p   = config["dropout_p"]
    lr          = config["lr"]
    lr_factor   = config["lr_factor"]
    lr_patience = config["lr_patience"]
    num_epochs  = config["num_epochs"]
    batch_size  = config["batch_size"]
    num_classes = config["num_classes"]

    set_seed(seed=1980 + session.get_world_rank())

    train_ds = session.get_dataset_shard("train")
    val_ds   = session.get_dataset_shard("val")

    llm = BertModel.from_pretrained(os.environ["BERT_MODEL"])
    embedding_dim = llm.config.hidden_size
    model = FinetunedLLM(llm, dropout_p, embedding_dim, num_classes)
    model = train.torch.prepare_model(model)

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=lr_factor, patience=lr_patience
    )

    batch_size_per_worker = batch_size // session.get_world_size()

    for epoch in range(num_epochs):
        train_loss = train_step(train_ds, batch_size_per_worker, model, num_classes, loss_fn, optimizer)
        val_loss, _, _ = eval_step(val_ds, batch_size_per_worker, model, num_classes, loss_fn)
        scheduler.step(val_loss)

        metrics = dict(epoch=epoch, lr=optimizer.param_groups[0]["lr"],
                       train_loss=train_loss, val_loss=val_loss)
        checkpoint = TorchCheckpoint.from_model(model=model)
        session.report(metrics, checkpoint=checkpoint)


In [21]:
# Train loop config
train_loop_config = {
    "dropout_p": 0.5,
    "lr": 1e-4,
    "lr_factor": 0.8,
    "lr_patience": 3,
    "num_epochs": 10,
    "batch_size": 256,
    "num_classes": num_classes,
}

In [22]:
resources_per_worker = {
    "CPU": 2,   
    "GPU": 0    
}
num_workers=2

scaling_config = ScalingConfig(
    num_workers=num_workers,
    use_gpu=bool(resources_per_worker["GPU"]),
    resources_per_worker=resources_per_worker,
)
 
# %%
checkpoint_config = CheckpointConfig(
    num_to_keep=1,
    checkpoint_score_attribute="val_loss",
    checkpoint_score_order="min",
)
run_config = RunConfig(
    name="llm",
    checkpoint_config=checkpoint_config,
    storage_path="~/ray_results",
)

In [23]:
train_ds=ray.data.read_parquet("local_data/train_v1")
val_ds=ray.data.read_parquet('local_data/test_v1')
train_ds=train_ds.materialize()
val_ds=val_ds.materialize()

Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-03-28 15:10:18,829	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 0.977.
2026-03-28 15:10:18,831	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 214813 rows


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-03-28 15:10:18,951	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 0.230.
2026-03-28 15:10:18,952	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 314739 rows
2026-03-28 15:10:18,961	INFO logging.py:392 -- Registered dataset logger for dataset dataset_7_0
2026-03-28 15:10:18,969	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_7_0. Full logs are in C:\Users\admin\AppData\Local\Temp\ray\session_2026-03-28_15-09-22_830872_9044\logs\ray-data
2026-03-28 15:10:18,971	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_7_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet]
2026-03-28 15:10:19,015	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_7_0 =======
2026-03-28 15:10:19,032	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-03-28 15:10:19,033	INFO logging_progress.py:227 -- Active & requested resources: 0/2 CPU, 0.0B/113.4MiB object store
2026-03-28 15:10:19,034	IN

In [24]:
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=train_loop_config,
    datasets={"train": train_ds, "val": val_ds},
    scaling_config=scaling_config,
    run_config=run_config,
)
 
# %%


In [ ]:
results=trainer.fit()

(TrainController pid=12744) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 2}] * 2
(TrainController pid=12744) [FailurePolicy] RETRY
(TrainController pid=12744)   Source: controller
(TrainController pid=12744)   Error count: 1 (max allowed: inf)
(TrainController pid=12744) Error: Training failed due to controller error:
(TrainController pid=12744) The worker group startup timed out after 60.0 seconds waiting for 2 workers. Potential causes include: (1) temporary insufficient cluster resources while waiting for autoscaling (ignore this warning in this case), (2) infeasible resource request where the provided `ScalingConfig` cannot be satisfied), and (3) transient network issues. Set the RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S environment variable to increase the timeout.
(TrainController pid=12744) Traceback (most recent call last):
(TrainController pid=12744)   File "c:\Users\admin\Documents\Projects\MadeWithML\.venv\Lib\site-packages\ray\trai